# Logbook

Team members:

- Carla Sophia Rodríguez Dander - A01781793
- Isaac Husny - A01027140
- Jesus Rodríguez - A01025112

Log Description:

This document provides a log detailing every step taken during the Liverpool Challenge, ranging from the initial data cleaning to the creation of a model designed to predict employee attrition.

This record aims to provide a clear description of the process followed during the development of the solution. Each phase of the project is addressed in detail, highlighting the decisions made, the methods employed, and the results obtained while building an effective model to anticipate voluntary employee departures.

## CSV Import

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
drive.mount('/content/drive')
path='/content/drive/Shareddrives/COLAB DATOS DE LIVERPOOL/Datos Liverpool/'
Historico=pd.read_csv(path+'RenunciasHistorico.csv')
Demo=pd.read_csv(path+'RenunciasDemo.csv')

Mounted at /content/drive


<ipython-input-1-edd372ffb39e>:6: DtypeWarning: Columns (5,11) have mixed types. Specify dtype option on import or set low_memory=False.
  Historico=pd.read_csv(path+'RenunciasHistorico.csv')


# Data selection and cleaning



## Functions








To simplify the workflow, we have grouped the following functions into a single code block. These functions serve as useful tools for manipulating and cleaning data within the DataFrame, addressing common tasks such as type conversion, duplicate identification, and the generation of statistical summaries.

In [ ]:
#1. Remplazo de valores str a valores int.
def stringaint(df, columna_origen, columna_destino):
  #Diccionario que mapea cada valor unico desde la columna de origen a su identificador int correspondiente de columna destino
  DiccMap = df.drop_duplicates().set_index(columna_origen)[columna_destino].to_dict()
  #Filtro en el diccionario de mapeo para retener solo valores enteros
  DiccMap={k: v for k, v in DiccMap.items() if isinstance(v, int)}
  # Reemplazamos valores string en la columna de clave por valores int
  df[columna_destino] = df.apply(lambda row: DiccMap.get(row[columna_origen], row[columna_destino]), axis=1)
  return df

#2. Identificación de valores duplicados.
def revisarDuplicados(df):
  #Localizamos todas las filas en donde los valores son los mismos.
  mask= (df.T == df.iloc[:, 0]).all()
  #Obtiene los índices de las filas con valores uniformes
  uniform_row_indices= df[mask].index.tolist()

  return len(uniform_row_indices), uniform_row_indices

#3. Frecuencia de ocurrencias
def crear_csv_ref(df, output_file_path, column1, column2):
    # Llenar valores NaN en las columnas con "Faltante"
    df[[column1, column2]] = df[[column1, column2]].fillna("Faltante")
    # Calcular la frecuencia de ocurrencias para cada par único de valores de columna1 y columna2
    df_counts = df.groupby([column1, column2]).size().reset_index(name='Conteo')
    # Reemplazar "Faltante" de vuelta a NaN si es necesario
    df_counts.replace("Faltante", np.nan, inplace=True)
    # Guardar el DataFrame en el nuevo archivo CSV
    df_counts.to_csv(output_file_path,encoding='utf-8', index=False)

#4. Eliminación de columnas especificas
def eliminar_cols_desc(df, columnas_a_eliminar):
  #Elimina las columnas especificadas
  if set(columnas_a_eliminar).issubset(df.columns):
    df.drop(columns=columnas_a_eliminar, inplace=True)
    #Guardamos el DataFrame modificado en el mismo archivo CSV
  else:
    print(f"¡Algunas columnas de {columnas_a_eliminar} no se encontraron en el CSV!")

#5. llenado de valores vacios
def fill_empty_values(df,col_name, text1):
  #Usamos fillna para llenar los valores vacíos en la segunda columna
  df[col_name].fillna(text1, inplace=True)
  #Guardamos en el mismo archivo
  print(f"Valores vacíos en la columna '{col_name}' han sido llenados con '{text1}'.")

#6. Convertir valores a mayuscula
def convert_column_to_uppercase(df, col_name):
    #Pasamos la columna especificada a mayúsculas
    df[col_name] = df[col_name].str.upper()

#7. Conversión y manejo de fechas, separación en columnas.
def split_date_column(df, date_column, year_col="year", month_col="month", day_col="day"):
    # Convert to string first to handle '9999-12-31' and other out-of-bounds dates
    df[date_column] = df[date_column].astype(str)
    # Replace out-of-bounds dates with a placeholder or NaN
    df[date_column] = df[date_column].replace('9999-12-31', pd.NA)
    # Now convert to datetime format
    df[date_column] = pd.to_datetime(df[date_column], errors='coerce')
    # Determine the location for the new columns
    col_loc = df.columns.get_loc(date_column)

    # Extract and cast columns to integer (if NaN values exist, they'll stay as floats)
    df.insert(col_loc + 1, year_col, df[date_column].dt.year.fillna(0).astype(int))
    df.insert(col_loc + 2, month_col, df[date_column].dt.month.fillna(0).astype(int))
    df.insert(col_loc + 3, day_col, df[date_column].dt.day.fillna(0).astype(int))

    return df

#8.Resume la distribución y el porcentaje de aparición de categorías en una columna del DataFrame.
def cuenta_por_cat(df,col):
  year_counts = df.groupby(col).size()
  total_rows = len(df)
  year_percentages = year_counts / total_rows * 100
  result = pd.DataFrame({
       'Conteo': year_counts,
       'Porcentaje': year_percentages
  })
  print(result)

#9. Eliminación de las filas NaT
def remove_nat_rows(df, date_column):
    return df[df[date_column].notna()]
def remove_rows_after_year(df, date_column, year):
    return df[df[date_column].dt.year <= year]

#10. Binarización de una columna
def genero_a_binario(genero):
    if genero == 'Hombre':
        return 0
    elif genero == 'Mujer':
        return 1
    else:
        return None
#11. Corrige códigos postales en una columna específica, añadiendo ceros para que tengan una longitud específica.
def correct_zipcode(df, column_name):
    for idx, row in df.iterrows():
        zipcode = str(row[column_name])
        if len(zipcode) == 4:
            df.at[idx, column_name] = zipcode + '0'
        elif len(zipcode) == 3:
            df.at[idx, column_name] = zipcode + '00'

    return df

## Function call...

### Preview of unmodified columns

In [ ]:
display(Historico.head(n=2))
display(Demo.head(n=2))


,Nº pers.,Hasta,Desde,Soc.,Desc Soc,DivP,Desc Div Per,GrPer,Desc Gpo Personal,ÁPers,Desc APers,Clave de organización,Ubicación,Un.org.,Desc Un Org,Función,Desc Función
0,3369,12/31/1998,12/1/1974,140,Servicios Liverpool S.A.,SELI,Servicios Liverpool,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,0,Sin Un Org,0,Sin Función
1,3369,12/31/1999,1/1/1999,140,Servicios Liverpool S.A.,SELI,Servicios Liverpool,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,40,Compras Juniors,83,Comprador Sr B


,Nº pers.,Fecha nacimiento,Genero,No Hijos,Fecha ingreso,Edad ingreso,CP Vivienda,CP Trabajo,Ubicación,Descubica,Fecha Salida,Antigüedad,Edad salida,Año salida,Desc Medida,Gpo Per,Desc Gpo Pers,Ultima Evaluación,Función,Desc Fun
0,70634620,1984-02-28,Hombre,0,2022-08-24,38,2770,234,986,Suburbia Los Cabos Patio,2022-10-08,0,38,2022,Baja Suburbia,O,SBB Planta,3.0,70000225,Jefe Prevencion Perdidas
1,70636679,1990-12-23,Mujer,0,2022-08-31,31,11500,234,986,Suburbia Los Cabos Patio,2022-11-03,0,31,2022,Baja Suburbia,O,SBB Planta,3.0,70000232,Supervisor Cajas


### Renaming columns


We modified the label designations to establish a more uniform and transparent coding system, thereby enhancing the understanding, maintenance, and utility of the information when categorizing and organizing data. We determined that the names originally used by Liverpool were less suitable for our needs, prompting us to rename them. This process significantly improves the effectiveness and efficiency of information management, facilitating the work of those who interact with these labels at various levels.

In [ ]:
Historico= Historico.rename(columns={'DivP':'Clave Ubicación',
                                     'Desc Div Per':'Locación',
                                     'Clave de organización': 'ID Depa',
                                     'Ubicación':'Departamento',
                                     ' Un.org.':'ID Unidad',
                                     'Desc Un Org':'Unidad',
                                     'ÁPers': 'ID Área Personal',
                                     'Desc APers':'Área Personal',
                                     'GrPer': 'ID GroPer',
                                     'Desc Gpo Personal':'GroPer'
                                     })

Demo= Demo.rename(columns={'Ubicación':'Clave Ubicación',
                           'Descubica':'Locación',
                           'Gpo Per': 'ID GroPer',
                           'Desc Gpo Pers':'GroPer'})

display(Historico.head(n=1))
display(Demo.head(n=1))

,Nº pers.,Hasta,Desde,Soc.,Desc Soc,Clave Ubicación,Locación,ID GroPer,GroPer,ID Área Personal,Área Personal,ID Depa,Departamento,ID Unidad,Unidad,Función,Desc Función
0,3369,12/31/1998,12/1/1974,140,Servicios Liverpool S.A.,SELI,Servicios Liverpool,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,0,Sin Un Org,0,Sin Función


,Nº pers.,Fecha nacimiento,Genero,No Hijos,Fecha ingreso,Edad ingreso,CP Vivienda,CP Trabajo,Clave Ubicación,Locación,Fecha Salida,Antigüedad,Edad salida,Año salida,Desc Medida,ID GroPer,GroPer,Ultima Evaluación,Función,Desc Fun
0,70634620,1984-02-28,Hombre,0,2022-08-24,38,2770,234,986,Suburbia Los Cabos Patio,2022-10-08,0,38,2022,Baja Suburbia,O,SBB Planta,3.0,70000225,Jefe Prevencion Perdidas


### From str to int

In the code presented below, we have developed a specific function designed to first identify all combinations involving text strings. Subsequently, the function converts these elements—initially represented as strings—into integer values. This approach was implemented to optimize data handling, enabling more efficient information manipulation and processing while ensuring a more consistent and appropriate representation for subsequent operations. This process significantly enhances the code's robustness and clarity, facilitating its understanding and maintenance across various contexts.

In [ ]:
stringaint(Historico, "Locación", "Clave Ubicación" )
stringaint(Historico, "Departamento", "ID Depa" )
stringaint(Historico, "Unidad", "ID Unidad" )
stringaint(Historico, "Desc Función", " Función" )

stringaint(Demo, "Locación", "Clave Ubicación" )
stringaint(Demo, "Desc Fun", "Función" )

display(Historico.head(n=2))
display(Demo.head(n=2))

,Nº pers.,Hasta,Desde,Soc.,Desc Soc,Clave Ubicación,Locación,ID GroPer,GroPer,ID Área Personal,Área Personal,ID Depa,Departamento,ID Unidad,Unidad,Función,Desc Función
0,3369,12/31/1998,12/1/1974,140,Servicios Liverpool S.A.,0,Servicios Liverpool,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,0,Sin Un Org,0,Sin Función
1,3369,12/31/1999,1/1/1999,140,Servicios Liverpool S.A.,0,Servicios Liverpool,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,40,Compras Juniors,83,Comprador Sr B


,Nº pers.,Fecha nacimiento,Genero,No Hijos,Fecha ingreso,Edad ingreso,CP Vivienda,CP Trabajo,Clave Ubicación,Locación,Fecha Salida,Antigüedad,Edad salida,Año salida,Desc Medida,ID GroPer,GroPer,Ultima Evaluación,Función,Desc Fun
0,70634620,1984-02-28,Hombre,0,2022-08-24,38,2770,234,986,Suburbia Los Cabos Patio,2022-10-08,0,38,2022,Baja Suburbia,O,SBB Planta,3.0,70000086,Jefe Prevencion Perdidas
1,70636679,1990-12-23,Mujer,0,2022-08-31,31,11500,234,986,Suburbia Los Cabos Patio,2022-11-03,0,31,2022,Baja Suburbia,O,SBB Planta,3.0,70000099,Supervisor Cajas


### CSV Data Optimization: Assignment and Filling
"CONSEJO" Key
We observed that the "ID Depa" column contains several cells with no values, and noted that the description associated with them is always "CONSEJO". Consequently, we decided to assign the key "0" to instances where the description is "CONSEJO".


Additionally, we identified that some department keys lack a specific assignment. Therefore, we will assign the label "Sin departamento" (No department) to the corresponding description in the *Departamentos.csv* file.


Regarding the implemented code, it fills empty values ​​in the second column of a CSV file with the text "Sin Departamento". To achieve this, the "fillna" function is used; it replaces empty values ​​in the second column (specified by "col_name") with the provided text ("text1"). Finally, the modified DataFrame is saved back to the same CSV file, and the last line of the code displays the updated DataFrame.


This procedure not only improves data integrity by assigning consistent values ​​where information is missing but also enhances clarity and consistency in data management within the dataset.

In [ ]:
Historico['ID Depa'].fillna(0, inplace=True)
display(Historico.head(n=2))
display(Demo.head(n=0))

,Nº pers.,Hasta,Desde,Soc.,Desc Soc,Clave Ubicación,Locación,ID GroPer,GroPer,ID Área Personal,Área Personal,ID Depa,Departamento,ID Unidad,Unidad,Función,Desc Función
0,3369,12/31/1998,12/1/1974,140,Servicios Liverpool S.A.,0,Servicios Liverpool,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,0,Sin Un Org,0,Sin Función
1,3369,12/31/1999,1/1/1999,140,Servicios Liverpool S.A.,0,Servicios Liverpool,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,40,Compras Juniors,83,Comprador Sr B


,Nº pers.,Fecha nacimiento,Genero,No Hijos,Fecha ingreso,Edad ingreso,CP Vivienda,CP Trabajo,Clave Ubicación,Locación,Fecha Salida,Antigüedad,Edad salida,Año salida,Desc Medida,ID GroPer,GroPer,Ultima Evaluación,Función,Desc Fun


In [ ]:
fill_empty_values(Historico, 'Departamento', 'SIN DEPTO')

Valores vacíos en la columna 'Departamento' han sido llenados con 'SIN DEPTO'.


### Duplicate values

In the code snippet below, we perform a check to identify rows containing duplicate values ​​across all columns. If such instances are found, we evaluate the option of removing them to maintain data integrity and consistency.


The code is designed to scan a CSV file and count rows where all values ​​match across every column. It then outputs the count of these rows along with a list of their respective indices.


It is worth noting that the analysis yielded a result of 0, indicating that no rows with fully duplicate values ​​were found. Consequently, no deletion was required. This process helps optimize data quality by mitigating potential duplication, thereby ensuring reliability and consistency in information management.

In [ ]:

numero_filas_uniformes, indices_filas_uniformes = revisarDuplicados(Historico)

print(f"Número de filas con valores uniformes: {numero_filas_uniformes}")
print("Índices de filas con valores uniformes:")

for indice in indices_filas_uniformes:
    print(indice)

Número de filas con valores uniformes: 0
Índices de filas con valores uniformes:


### Extraction of small DataFrames

In the following code block, we have implemented the creation of additional CSV files to store descriptions of columns containing keys, for future reference.

In this code, both the columns and the file paths have been defined for each call to the "crear_csv_ref" function. A *for* loop is then used to extract the data and save it to the corresponding CSV files, with an informational message printed after each operation. This approach provides a more organized structure and makes the process easier to understand.

Upon completion, a message indicates that the process has finished successfully. This improvement not only clarifies the code's output but also makes the actions performed easier to grasp, contributing to a smoother and more understandable experience for anyone reviewing the code in the future.

In [ ]:
# Definir las columnas y rutas de archivo
columnas_y_rutas = [
    ("Clave Ubicación", "Locación", "Locaciones.csv"),
    ("Soc.", "Desc Soc", "Sociedades.csv"),
    ("ID GroPer", "GroPer", "Grupo_Personas.csv"),
    ("ID Área Personal", "Área Personal", "Area Personal.csv"),
    ("ID Depa", "Departamento", "Departamentos.csv"),
    ("ID Unidad", "Unidad", "Unidades.csv"),
    (" Función", "Desc Función", "Funciones.csv")]

# Realizar la extracción de la lista anterior y guardar en archivos CSV
for column1, column2, output_file in columnas_y_rutas:
    df = Historico
    crear_csv_ref(df, path + output_file, column1, column2)
    print(f"Se ha extraído y guardado '{column1}' y '{column2}' en '{output_file}'")

print("Proceso completado.")


Se ha extraído y guardado 'Clave Ubicación' y 'Locación' en 'Locaciones.csv'
Se ha extraído y guardado 'Soc.' y 'Desc Soc' en 'Sociedades.csv'
Se ha extraído y guardado 'ID GroPer' y 'GroPer' en 'Grupo_Personas.csv'
Se ha extraído y guardado 'ID Área Personal' y 'Área Personal' en 'Area Personal.csv'
Se ha extraído y guardado 'ID Depa' y 'Departamento' en 'Departamentos.csv'
Se ha extraído y guardado 'ID Unidad' y 'Unidad' en 'Unidades.csv'
Se ha extraído y guardado ' Función' y 'Desc Función' en 'Funciones.csv'
Proceso completado.


### Date Format

The **Hasta** column was converted to the MM-DD-YYYY format, and separate columns were subsequently added to incorporate detailed information regarding the **day, month, year, hour, and minute**. The complexity of this procedure stemmed from the presence of seconds and NaN values ​​within the date data.

To address this, we implemented the **pd.to_datetime** function with *errors='coerce'* to handle the dates and apply the required format. Subsequently, the various date and time components were extracted using regular expressions. These components were converted to integers and assigned to their respective columns.

This approach not only optimized the representation of dates in the desired format but also resolved issues related to the presence of seconds and null values. The use of regular expressions adds an extra layer of precision to the extraction process, enhancing the quality and consistency of the data in the transformed dataset.

In [ ]:
split_date_column(Historico, 'Hasta', 'HAño', 'HMes', 'HDia')
split_date_column(Historico, 'Desde', 'DAño', 'DMes', 'DDia')
split_date_column(Demo, 'Fecha ingreso', 'IAño', 'IMes', 'IDia')
split_date_column(Demo, 'Fecha Salida', 'SAño', 'SMes', 'SDia')

,Nº pers.,Fecha nacimiento,Genero,No Hijos,Fecha ingreso,IAño,IMes,IDia,Edad ingreso,CP Vivienda,...,SDia,Antigüedad,Edad salida,Año salida,Desc Medida,ID GroPer,GroPer,Ultima Evaluación,Función,Desc Fun
0,70634620,1984-02-28,Hombre,0,2022-08-24,2022,8,24,38,2770,...,8,0,38,2022,Baja Suburbia,O,SBB Planta,3.0,70000086,Jefe Prevencion Perdidas
1,70636679,1990-12-23,Mujer,0,2022-08-31,2022,8,31,31,11500,...,3,0,31,2022,Baja Suburbia,O,SBB Planta,3.0,70000099,Supervisor Cajas
2,70642442,1981-06-20,Mujer,0,2022-09-28,2022,9,28,41,23410,...,22,0,41,2022,Baja Suburbia,O,SBB Planta,3.0,70000103,Auxiliar Cajero
3,70639827,1993-05-02,Hombre,0,2022-09-14,2022,9,14,29,23428,...,26,0,29,2022,Baja Suburbia,O,SBB Planta,3.0,70000098,Subjefe Proteccion
4,13111365,1993-08-29,Mujer,0,2014-11-13,2014,11,13,21,23450,...,14,7,29,2022,Baja Suburbia,O,SBB Planta,3.0,0,Sin Función
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117155,15178424,1994-10-04,Hombre,0,2019-10-28,2019,10,28,25,Sin Dato,...,30,0,25,2019,Baja,D,Vía Planta No Sind.,3.0,823,Vendedor Boutique Sfera
117156,15200769,1997-09-29,Mujer,0,2019-11-13,2019,11,13,22,Sin Dato,...,1,0,22,2020,Baja,D,Vía Planta No Sind.,3.0,823,Vendedor Boutique Sfera
117157,15273135,1999-09-18,Hombre,0,2020-02-17,2020,2,17,20,Sin Dato,...,20,0,20,2020,Baja,D,Vía Planta No Sind.,3.0,823,Vendedor Boutique Sfera
117158,15280203,1990-12-09,Mujer,0,2020-02-26,2020,2,26,29,Sin Dato,...,13,0,29,2020,Baja,D,Vía Planta No Sind.,3.0,823,Vendedor Boutique Sfera


In [ ]:
display(Historico.head(n=2))

,Nº pers.,Hasta,HAño,HMes,HDia,Desde,DAño,DMes,DDia,Soc.,...,ID GroPer,GroPer,ID Área Personal,Área Personal,ID Depa,Departamento,ID Unidad,Unidad,Función,Desc Función
0,3369,1998-12-31,1998,12,31,1974-12-01,1974,12,1,140,...,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,0,Sin Un Org,0,Sin Función
1,3369,1999-12-31,1999,12,31,1999-01-01,1999,1,1,140,...,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,40,Compras Juniors,83,Comprador Sr B


## Order

### Optimization of Employment Information: Introduction of Duration-in-Years Columns into Historical and Demographic Data

An additional column has been added to both the historical data and demographic data documents to reflect employees' length of service in years. Including this column in the demographic file is optional, as that file already contains a column providing information on employee tenure.

This enhancement facilitates the assessment and analysis of employee tenure, offering a time-based indicator that can be crucial for understanding workforce dynamics. The additional column in the historical data file provides relevant temporal context, whereas its inclusion in the demographic file is considered optional due to the presence of a pre-existing column that serves a similar purpose.







#### Column Duracion DEMO

In [ ]:
display(Demo.head(n=2))

,Nº pers.,Fecha nacimiento,Genero,No Hijos,Fecha ingreso,IAño,IMes,IDia,Edad ingreso,CP Vivienda,...,SDia,Antigüedad,Edad salida,Año salida,Desc Medida,ID GroPer,GroPer,Ultima Evaluación,Función,Desc Fun
0,70634620,1984-02-28,Hombre,0,2022-08-24,2022,8,24,38,2770,...,8,0,38,2022,Baja Suburbia,O,SBB Planta,3.0,70000086,Jefe Prevencion Perdidas
1,70636679,1990-12-23,Mujer,0,2022-08-31,2022,8,31,31,11500,...,3,0,31,2022,Baja Suburbia,O,SBB Planta,3.0,70000099,Supervisor Cajas


In [ ]:
Demo['Fecha Salida'] = pd.to_datetime(Demo['Fecha Salida'])
# Calcula la duración en una nueva columna llamada 'Duracion'
Demo['Duracion'] = (Demo['Fecha Salida'] - Demo['Fecha ingreso']).dt.days/365
# Agrega la columna de duración al lado de la columna 'DDia'
#Demo = pd.concat([Demo['SDia'], Demo['Duracion'], Demo.drop(['SDia', 'Duracion'], axis=1)], axis=1)
column_orderD= ['Nº pers.',	'Fecha nacimiento',	'Genero','No Hijos', 'Fecha ingreso','IAño',	'IMes',
                'Edad ingreso',	'Fecha Salida',	'SAño',	'SMes','Duracion', 'Antigüedad','Edad salida',	'Año salida',
                'Desc Medida',	'GroPer',	'Ultima Evaluación', 'CP Vivienda','CP Trabajo','Locación','Desc Fun']

Demo=Demo[column_orderD]
display(Demo.head(n=2))

,Nº pers.,Fecha nacimiento,Genero,No Hijos,Fecha ingreso,IAño,IMes,Edad ingreso,Fecha Salida,SAño,...,Antigüedad,Edad salida,Año salida,Desc Medida,GroPer,Ultima Evaluación,CP Vivienda,CP Trabajo,Locación,Desc Fun
0,70634620,1984-02-28,Hombre,0,2022-08-24,2022,8,38,2022-10-08,2022,...,0,38,2022,Baja Suburbia,SBB Planta,3.0,2770,234,Suburbia Los Cabos Patio,Jefe Prevencion Perdidas
1,70636679,1990-12-23,Mujer,0,2022-08-31,2022,8,31,2022-11-03,2022,...,0,31,2022,Baja Suburbia,SBB Planta,3.0,11500,234,Suburbia Los Cabos Patio,Supervisor Cajas


#### Column Duracion HISTORICO

In [ ]:
display(Historico.head(n=2))

,Nº pers.,Hasta,HAño,HMes,HDia,Desde,DAño,DMes,DDia,Soc.,...,ID GroPer,GroPer,ID Área Personal,Área Personal,ID Depa,Departamento,ID Unidad,Unidad,Función,Desc Función
0,3369,1998-12-31,1998,12,31,1974-12-01,1974,12,1,140,...,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,0,Sin Un Org,0,Sin Función
1,3369,1999-12-31,1999,12,31,1999-01-01,1999,1,1,140,...,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,40,Compras Juniors,83,Comprador Sr B


In [ ]:
Historico['Desde'] = pd.to_datetime(Historico['Desde'])
Historico['Hasta'] = pd.to_datetime(Historico['Hasta'])
# Calcula la duración en una nueva columna llamada 'Duracion'
Historico['Duracion'] = (Historico['Hasta'] - Historico['Desde']).dt.days/365
# Agrega la columna de duración al lado de la columna 'DDia'
#Historico = pd.concat([Historico[], Historico['Duracion'], Historico.drop(['Duracion'], axis=1)], axis=1)
column_orderH= ['Nº pers.', 'Hasta', 'HAño', 'HMes', 'Desde', 'DAño', 'DMes',
                'Duracion', 'Desc Soc','Locación', 'GroPer','Área Personal',
                'Departamento', 'Unidad', 'Desc Función']
Historico=Historico[column_orderH]
display(Historico.head(n=2))

,Nº pers.,Hasta,HAño,HMes,Desde,DAño,DMes,Duracion,Desc Soc,Locación,GroPer,Área Personal,Departamento,Unidad,Desc Función
0,3369,1998-12-31,1998,12,1974-12-01,1974,12,24.09863,Servicios Liverpool S.A.,Servicios Liverpool,Planta No Sind.,Ejecutivo/ Coord.,JUNIORS,Sin Un Org,Sin Función
1,3369,1999-12-31,1999,12,1999-01-01,1999,1,0.99726,Servicios Liverpool S.A.,Servicios Liverpool,Planta No Sind.,Ejecutivo/ Coord.,JUNIORS,Compras Juniors,Comprador Sr B


### Remove NaN rows and columns and NaT values

#### Rows and columns with NaN
We removed all rows containing NaN values ​​using the `dropna` function. A subsequent count revealed no rows meeting this criterion, demonstrating the data's robustness regarding row integrity.

Next, we proceeded to remove columns where all values ​​were NaN. Upon performing this operation and counting the results, we found no columns meeting this condition, highlighting the data's consistency in terms of column integrity. Consequently, no columns were removed at this stage. This process contributes significantly to data cleaning and quality assurance, ensuring the dataset maintains its coherence and utility.

In [ ]:
Historico.dropna(how='all', inplace=True)
num_filas_despues = len(Historico.index)
# Imprimir el número de filas antes y después de la eliminación
print("Número de filas antes de eliminar NaN en Historico:", len(Historico.index))
print("Número de filas después de eliminar NaN en Historico:", num_filas_despues)

Demo.dropna(how='all', inplace=True)
num_filas_despues = len(Demo.index)
# Imprimir el número de filas antes y después de la eliminación
print("Número de filas antes de eliminar NaN en Demo:", len(Demo.index))
print("Número de filas después de eliminar NaN en Demo:", num_filas_despues)

Número de filas antes de eliminar NaN en Historico: 560130
Número de filas después de eliminar NaN en Historico: 560130
Número de filas antes de eliminar NaN en Demo: 117160
Número de filas después de eliminar NaN en Demo: 117160


<ipython-input-16-42f5f1706817>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Historico.dropna(how='all', inplace=True)
<ipython-input-16-42f5f1706817>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Demo.dropna(how='all', inplace=True)


In [ ]:
print("# Columnas antes en Doc Historico:",len(Historico.columns))
Historico.dropna(how='all', axis=1, inplace=True)
#Numero de filas en el dataframe después de la operación
print("# Columnas despues en Doc Historico:",len(Historico.columns))

print("# Columnas antes en Doc Demo:",len(Demo.columns))
Demo.dropna(how='all', axis=1, inplace=True)
#Numero de filas en el dataframe después de la operación
print("# Columnas despues en Doc Demo:",len(Demo.columns))

# Columnas antes en Doc Historico: 15
# Columnas despues en Doc Historico: 15
# Columnas antes en Doc Demo: 22
# Columnas despues en Doc Demo: 22


<ipython-input-17-67e706d15fe5>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Historico.dropna(how='all', axis=1, inplace=True)
<ipython-input-17-67e706d15fe5>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Demo.dropna(how='all', axis=1, inplace=True)


#### NaT Values
The "remove_nat_rows" function was used to eliminate rows with NaT values ​​in the 'Hasta' column of the "Historico" DataFrame, thereby improving data integrity. Additionally, "remove_rows_after_year" was employed to discard rows where the 'Hasta' value is later than 2023, focusing the analysis on relevant data up to that year.

In summary, these lines of code are designed to enhance the quality and relevance of the data in the "Historico" DataFrame by removing both rows with NaT values ​​and those containing information beyond 2023 in the 'Hasta' column. This procedure is essential for obtaining a refined dataset focused on the desired timeframe.

In [ ]:
Historico = remove_nat_rows(Historico, 'Hasta')
Historico = remove_rows_after_year(Historico, 'Hasta', 2023)

In [ ]:
display(Historico.head(n=2))
display(Demo.head(n=2))

,Nº pers.,Hasta,HAño,HMes,Desde,DAño,DMes,Duracion,Desc Soc,Locación,GroPer,Área Personal,Departamento,Unidad,Desc Función
0,3369,1998-12-31,1998,12,1974-12-01,1974,12,24.09863,Servicios Liverpool S.A.,Servicios Liverpool,Planta No Sind.,Ejecutivo/ Coord.,JUNIORS,Sin Un Org,Sin Función
1,3369,1999-12-31,1999,12,1999-01-01,1999,1,0.99726,Servicios Liverpool S.A.,Servicios Liverpool,Planta No Sind.,Ejecutivo/ Coord.,JUNIORS,Compras Juniors,Comprador Sr B


,Nº pers.,Fecha nacimiento,Genero,No Hijos,Fecha ingreso,IAño,IMes,Edad ingreso,Fecha Salida,SAño,...,Antigüedad,Edad salida,Año salida,Desc Medida,GroPer,Ultima Evaluación,CP Vivienda,CP Trabajo,Locación,Desc Fun
0,70634620,1984-02-28,Hombre,0,2022-08-24,2022,8,38,2022-10-08,2022,...,0,38,2022,Baja Suburbia,SBB Planta,3.0,2770,234,Suburbia Los Cabos Patio,Jefe Prevencion Perdidas
1,70636679,1990-12-23,Mujer,0,2022-08-31,2022,8,31,2022-11-03,2022,...,0,31,2022,Baja Suburbia,SBB Planta,3.0,11500,234,Suburbia Los Cabos Patio,Supervisor Cajas


### Remove columns from the DataFrame
Once the data cleaning phase was complete, we generated CSV files containing the various keys for each concept, their respective descriptions, and the post-cleaning occurrence counts. This approach provides a detailed and organized view of the essential information following the data refinement process.

Subsequently, we removed the columns that had been saved to the new CSV files. This measure was implemented to improve the organization of the DataFrame by eliminating columns that were already stored independently. This strategy helps maintain a clearer and more efficient DataFrame structure while preserving the essential information in the generated CSV files.

In [ ]:
eliminar_cols_desc(Historico, ["Soc.", "ID GroPer","ID Área Personal", "ID Depa", "Unidad"])
eliminar_cols_desc(Demo,["GroPer"])
display(Historico.head(n=4))
display(Demo.head(n=4))

¡Algunas columnas de ['Soc.', 'ID GroPer', 'ID Área Personal', 'ID Depa', 'Unidad'] no se encontraron en el CSV!


<ipython-input-2-85105484c5be>:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=columnas_a_eliminar, inplace=True)


,Nº pers.,Hasta,HAño,HMes,Desde,DAño,DMes,Duracion,Desc Soc,Locación,GroPer,Área Personal,Departamento,Unidad,Desc Función
0,3369,1998-12-31,1998,12,1974-12-01,1974,12,24.098630,Servicios Liverpool S.A.,Servicios Liverpool,Planta No Sind.,Ejecutivo/ Coord.,JUNIORS,Sin Un Org,Sin Función
1,3369,1999-12-31,1999,12,1999-01-01,1999,1,0.997260,Servicios Liverpool S.A.,Servicios Liverpool,Planta No Sind.,Ejecutivo/ Coord.,JUNIORS,Compras Juniors,Comprador Sr B
2,3369,2004-01-31,2004,1,2000-01-01,2000,1,4.084932,Servicios Liverpool S.A.,Servicios Liverpool,Planta No Sind.,Ejecutivo/ Coord.,JUNIORS,Compras Juniors,Comprador Sr B
3,3369,2005-11-30,2005,11,2004-02-01,2004,2,1.830137,Servicios Liverpool S.A.,Servicios Liverpool,Planta No Sind.,Ejecutivo/ Coord.,JUNIORS,Compras Juniors,Comprador Sr A


,Nº pers.,Fecha nacimiento,Genero,No Hijos,Fecha ingreso,IAño,IMes,Edad ingreso,Fecha Salida,SAño,...,Duracion,Antigüedad,Edad salida,Año salida,Desc Medida,Ultima Evaluación,CP Vivienda,CP Trabajo,Locación,Desc Fun
0,70634620,1984-02-28,Hombre,0,2022-08-24,2022,8,38,2022-10-08,2022,...,0.123288,0,38,2022,Baja Suburbia,3.0,2770,234,Suburbia Los Cabos Patio,Jefe Prevencion Perdidas
1,70636679,1990-12-23,Mujer,0,2022-08-31,2022,8,31,2022-11-03,2022,...,0.175342,0,31,2022,Baja Suburbia,3.0,11500,234,Suburbia Los Cabos Patio,Supervisor Cajas
2,70642442,1981-06-20,Mujer,0,2022-09-28,2022,9,41,2022-11-22,2022,...,0.150685,0,41,2022,Baja Suburbia,3.0,23410,234,Suburbia Los Cabos Patio,Auxiliar Cajero
3,70639827,1993-05-02,Hombre,0,2022-09-14,2022,9,29,2022-10-26,2022,...,0.115068,0,29,2022,Baja Suburbia,3.0,23428,234,Suburbia Los Cabos Patio,Subjefe Proteccion


## Merge

### Merge between Histórico y Demo

We carried out a comprehensive process to optimize information regarding employees' work history. First, we renamed the 'Hasta' column to 'Fecha Salida' (Exit Date) in the "Historico" DataFrame to improve clarity. Next, we organized the DataFrame by employee number ('Nº pers.') and exit date ('Fecha Salida') and selected the first row for each employee number, thereby identifying each employee's most recent position. This approach is beneficial as it provides a clearer representation of each employee's latest work history, facilitating the analysis of trends and changes over time.

Subsequently, during the merging phase, we combined information from the "Demo" DataFrame with the data regarding each employee's latest position extracted from the "Historico" DataFrame. This step was performed using the `pd.merge` function, with the employee number ('Nº pers.') serving as the join key. Selecting specific columns from the "ultimo_puesto" DataFrame provided essential details—such as personnel group ('GroPer'), personnel area ('Área Personal'), department, and unit—which were incorporated into the resulting "merged_df" DataFrame.

This integration is valuable for gaining a more complete view of the workforce, enabling a deeper analysis of the organizational structure and employees' career paths. The length of the resulting DataFrame offers a quantitative measure of the number of records, providing a quick assessment of how these operations affected the overall dataset. In summary, this process contributes to more effective workforce data management, facilitating more accurate insights and supporting informed decision-making within the organization.


In [ ]:
Historico = Historico.rename(columns={'Hasta': 'Fecha Salida'})
# Ordena el DataFrame por Nº pers. y Fecha de egreso de forma descendente
df_sorted = Historico.sort_values(by=['Nº pers.', 'Fecha Salida'], ascending=[True, False])

# Selecciona la primera fila para cada Nº pers., que ahora será el último puesto
ultimo_puesto = df_sorted.drop_duplicates(subset='Nº pers.', keep='first')    #elimina los valores duplicados en ID

merged_df = pd.merge(Demo, ultimo_puesto[['Nº pers.', 'GroPer', 'Área Personal', 'Departamento', 'Unidad']],    #mezclamos ambos dataframes (en base al ID del empleado)
                     on=['Nº pers.'], how='left')

len(merged_df)

117160

>These lines of code aim to assess and report the presence of null values ​​in the 'Departamento' column of the "merged_df" DataFrame. We use **.isna()** to identify null values ​​in that column and then **.sum()** to count how many such values ​​exist. The result is stored in the variable "valores_nulos". Subsequently, the number of null values ​​in the 'Departamento' column is printed to ensure there is no missing data following the DataFrame merge, thereby verifying the integrity of the consolidated data.

In [ ]:
# Cuenta los valores nulos en la columna 'Nombre_Columna'
valores_nulos = merged_df['Departamento'].isna().sum()

# Imprime la cantidad de valores nulos
print('Cantidad de valores nulos en la columna:', valores_nulos)    #nos aseguramos de que no haya valores nulos tras hacer el merge

Cantidad de valores nulos en la columna: 4


>In these lines, we are removing rows from the `merged_df` DataFrame that contain missing values ​​(NaN) in the column specified as 'Departamento'. The `dropna` function is used with the argument `subset=['Departamento']`, indicating that rows with missing values ​​should be removed only in the 'Departamento' column. The parameter `inplace=True` means that these changes are applied directly to the DataFrame—modifying the original one—without the need to create a new copy. This process ensures that only rows with complete information in the 'Departamento' column are retained, thereby contributing to data integrity and consistency for subsequent analysis.

In [ ]:
# Drop rows with missing values in the specified column
merged_df.dropna(subset=['Departamento'], inplace=True)

### Removal of "Sin Función" string in 'Desc Fun' column

To eliminate data that essentially acts as noise during modeling, a decision was made to remove all records lacking a **function**. This process was applied to both dataframes—the merged dataset and the historical dataset—to ensure the latter could also be modeled independently if needed.

The code performs a series of operations on the `merged_df` DataFrame. First, it counts the rows in the original DataFrame before any filtering takes place, establishing an initial baseline. Next, it filters the DataFrame to exclude rows where the 'Desc Fun' column contains the string 'Sin Función' (regardless of capitalization). Following this filter, the rows are counted again to determine how many were retained. By calculating the difference between the original and new counts, the total number of removed rows is determined. Finally, a message is printed to clearly and directly report the number of rows removed during this filtering process on `merged_df`. In summary, these lines provide a quantitative view of the filtering impact on the dataset, highlighting the number of rows removed based on the specified criterion.

In [ ]:
# Contar el número de filas antes de la eliminación en Demo
num_filas_demo_original = len(merged_df)

# Filtrar filas en las que 'Desc Fun' no contiene 'Sin Función' en el DataFrame Demo
merged_df = merged_df[~merged_df['Desc Fun'].str.contains('Sin Función', case=False, na=False)]

# Contar el número de filas después de la eliminación en Demo
num_filas_demo_filtrado = len(merged_df)

# Calcular el número de filas eliminadas en Demo
filas_eliminadas_demo = num_filas_demo_original - num_filas_demo_filtrado

# Mostrar el número de filas eliminadas en Demo
print(f'Se han eliminado {filas_eliminadas_demo} filas del DataFrame merged_df.')

Se han eliminado 1443 filas del DataFrame merged_df.


In [ ]:
# Contar el número de filas antes de la eliminación en Historico
num_filas_historico_original = len(Historico)

# Filtrar filas en las que 'Desc Fun' no contiene 'Sin Función' en el DataFrame Historico
Historico = Historico[~Historico['Desc Función'].str.contains('Sin Función', case=False, na=False)]

# Contar el número de filas después de la eliminación en Historico
num_filas_historico_filtrado = len(Historico)

# Calcular el número de filas eliminadas en Historico
filas_eliminadas_historico = num_filas_historico_original - num_filas_historico_filtrado

# Mostrar el número de filas eliminadas en Historico
print(f'Se han eliminado {filas_eliminadas_historico} filas del DataFrame Historico.')


Se han eliminado 6726 filas del DataFrame Historico.


### Remove workers with less than 4 months of service

To reduce noise and improve data quality for future models, the decision was made to exclude all employees with a tenure of less than four months. This decision is based on the premise that data from very short periods could introduce unrepresentative variability and negatively impact future predictions.

>These lines of code add a time metric, 'Duración Días', to the "merged_df" and "Historico" DataFrames, calculating the difference in days between the exit and entry dates for each employee. This step facilitates the analysis of the duration of employees' tenure within the organization.

In [ ]:
# Calcular la duración en días en el DataFrame Demo
merged_df['Duración Días'] = (pd.to_datetime(merged_df['Fecha Salida']) - pd.to_datetime(merged_df['Fecha ingreso'])).dt.days

# Calcular la duración en días en el DataFrame Historico
Historico['Duración Días'] = (pd.to_datetime(Historico['Fecha Salida']) - pd.to_datetime(Historico['Desde'])).dt.days

>These lines of code calculate and display the number of rows in the "merged_df" and "Historico" DataFrames before performing deletion operations. These values ​​provide an initial reference regarding the original size of the datasets, serving as a useful basis for evaluating the impact of future modifications.

In [ ]:
# Contar el número de filas antes de la eliminación en Demo
num_filas_demo_original = len(merged_df)

# Contar el número de filas antes de la eliminación en Historico
num_filas_historico_original = len(Historico)

print(num_filas_demo_original)
print(num_filas_historico_original)

115713
436211


These lines of code perform a filtering process on the "merged_df" and "Historico" DataFrames to select only those rows that meet a specific duration criterion. Specifically, only rows with a duration exceeding 121 days (4 months) are retained in both datasets.

Next, the number of rows removed from each DataFrame is calculated by comparing the original length with the length resulting from the filtering process. The difference between the original row count and the current count represents the number of rows eliminated from each dataset.

Finally, the number of removed rows is printed for both the "Demo" and "Historico" DataFrames. This process yields quantitative data on the number of records failing to meet the established duration criterion, offering insight into the effectiveness of the filtering applied to each dataset.

In [ ]:
# Filtrar las filas en Demo con una duración mayor a 121 días
merged_df = merged_df[merged_df['Duración Días'] > 121]
# Filtrar las filas en Historico con una duración mayor a 121 días
Historico = Historico[Historico['Duración Días'] > 121]

# Calcular el número de filas eliminadas en Demo
filas_eliminadas_demo = num_filas_demo_original - len(merged_df)
# Calcular el número de filas eliminadas en Historico
filas_eliminadas_historico = num_filas_historico_original - len(Historico)


# Mostrar el número de filas eliminadas en Demo
print(f'Se han eliminado {filas_eliminadas_demo} filas del DataFrame Demo.')
# Mostrar el número de filas eliminadas en Historico
print(f'Se han eliminado {filas_eliminadas_historico} filas del DataFrame Historico.')

Se han eliminado 53821 filas del DataFrame Demo.
Se han eliminado 277448 filas del DataFrame Historico.


### Modification of time columns


This section focuses on optimizing the processing of temporal data for more effective application in modeling, thereby enhancing our ability to assess prediction accuracy with greater precision. This more efficient approach not only improves the quality of temporal analysis but also allows for greater sensitivity in evaluating predictive accuracy. Ultimately, these adjustments contribute significantly to a more holistic and detailed understanding of employees' work histories, thereby enriching data quality and strengthening the foundation for future analyses and projections.

#### Create a new column 'Duración Meses'

In the code, the `merged_df` DataFrame is manipulated to handle dates and incorporate temporal information. The 'Fecha Salida' and 'Fecha ingreso' columns are converted to datetime format. Next, the duration of each job in months is calculated, with the result stored in a new column named 'Duración Meses'. Finally, the "Demo" DataFrame is updated with the transformations applied to `merged_df`, enabling a more precise analysis of the temporal data within the employees' work history. Overall, these operations contribute to efficiently improving and enriching the information.

In [ ]:
# Convert the 'date_column' to datetime
merged_df['Fecha Salida'] = pd.to_datetime(merged_df['Fecha Salida'])

# Convert the 'date_column' to datetime
merged_df['Fecha ingreso'] = pd.to_datetime(merged_df['Fecha ingreso'])

merged_df['Duración Meses'] = (merged_df['Fecha Salida'].dt.year - merged_df['Fecha ingreso'].dt.year) * 12 + (merged_df['Fecha Salida'].dt.month - merged_df['Fecha ingreso'].dt.month)

Demo = merged_df

#### Create a new column 'Duración Trimestres'

These lines of code introduce a significant transformation to the "Demo" DataFrame. A new column named 'Duración Trimestres' is created, calculating the number of quarters equivalent to the duration in months for each entry in the 'Duración Meses' column. This calculation utilizes the `np.ceil` function—which rounds up to the nearest integer—followed by a conversion to the integer data type. In essence, this operation provides a measure of each entry's duration in quarters, offering a more granular temporal perspective and facilitating analyses that require a more detailed subdivision of time.

In [ ]:
# Crear la nueva columna 'Duración Trimestres' redondeando al entero más cercano
Demo['Duración Trimestres'] = np.ceil(Demo['Duración Meses'] / 3).astype(int)

In [ ]:
display(merged_df)

,Nº pers.,Fecha nacimiento,Genero,No Hijos,Fecha ingreso,IAño,IMes,Edad ingreso,Fecha Salida,SAño,...,CP Trabajo,Locación,Desc Fun,GroPer,Área Personal,Departamento,Unidad,Duración Días,Duración Meses,Duración Trimestres
19,13307867,1997-03-13,Mujer,0,2015-06-26,2015,6,18,2022-12-14,2022,...,234,Suburbia Los Cabos Patio,Cajero,SBB Planta,SBB Operación,CAJAS,Jefatura Cajas SBB Patio,2728,90,30
20,14877876,1991-05-14,Hombre,0,2019-02-01,2019,2,27,2021-10-16,2021,...,234,Boutique Los Cabos,Vendedor Bilingue,Planta No Sind.,TC. Vendedor,FRAGANCIAS,Ventas Boutique Los Cabos,988,32,11
21,15228049,1992-03-17,Mujer,2,2019-12-01,2019,12,27,2022-12-07,2022,...,234,Boutique Los Cabos,Consejero de Belleza,Vía Planta No Sind.,Personal General,FRAGANCIAS,Boutique Cabo San Lucas,1102,36,12
22,12379884,1970-03-31,Mujer,0,2012-09-20,2012,9,42,2020-02-01,2020,...,234,Boutique Los Cabos,Vendedor Bilingue,Planta No Sind.,TC. Vendedor,FRAGANCIAS,Ventas Boutique Los Cabos,2690,89,30
23,14559473,1965-01-10,Mujer,0,2018-05-15,2018,5,53,2022-11-04,2022,...,234,Boutique Los Cabos,Consejero de Belleza,Planta No Sind.,Personal General,FRAGANCIAS,Boutique Cabo San Lucas,1634,54,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117144,14503279,1996-07-18,Hombre,0,2018-04-04,2018,4,21,2019-04-04,2019,...,5109,Sfera Tlaquepaque,Vendedor Boutique Sfera,Planta No Sind.,MT. Vendedor,BOUTIQUES,Ventas Sfera BT Tlaquepaq,365,12,4
117145,14583940,1996-01-01,Mujer,0,2018-06-06,2018,6,22,2019-01-29,2019,...,5348,Sfera Playa del Carmen,Vendedor Boutique Sfera,Planta No Sind.,MT. Vendedor,BOUTIQUES,Ventas Sfera BT Playa del,237,7,3
117149,14864493,1990-08-30,Hombre,0,2019-01-18,2019,1,28,2020-01-14,2020,...,5348,Sfera Guadalajara Gran Plaza,Vendedor Boutique Sfera,Planta No Sind.,TC. Vendedor,BOUTIQUES,Ventas Sfera Gran Plaza,361,12,4
117153,15099771,1998-05-26,Hombre,0,2019-09-02,2019,9,21,2020-01-28,2020,...,5109,Sfera Galerias Lag Torreon,Vendedor Boutique Sfera,Planta No Sind.,MT. Vendedor,BOUTIQUES,Ventas Sfera BT Gal Lagun,148,4,2


### Removal of Outliers in Numeric Columns




In [ ]:
#Columna 'Antigüedad'
column_name = 'Antigüedad'
column_data = Demo[column_name]

# Calculate quartiles and IQR
Q1 = column_data.quantile(0.5)
Q3 = column_data.quantile(0.95)
IQR = Q3 - Q1

# Define the upper and lower bounds for outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Create a new DataFrame without outliers
Demo = Demo[(column_data >= lower_bound) & (column_data <= upper_bound)]

In [ ]:
#Columna 'SAño'
# Filter rows where the specified integer value is not present in the specified column
Demo = Demo[Demo['SAño'] != 2023]

0    42136
1     8626
2     6673
3     3208
4      610
5      118
6       26
7        7
Name: No Hijos, dtype: int64

In [ ]:
# Step 1: Count the occurrences of each unique string value
value_counts = Demo['No Hijos'].value_counts()

# Step 2: Filter string values with a count of less than 100
strings_to_drop = value_counts[value_counts < 200].index

# Step 3: Use boolean indexing to drop rows with these string values
Demo = Demo[~Demo['No Hijos'].isin(strings_to_drop)]

In [ ]:
#Columna 'Edad ingreso'
column_name = 'Edad ingreso'
column_counts = Demo[column_name].value_counts()
column_counts

19    4881
18    4679
20    4444
22    4177
21    4162
23    4017
24    3670
25    3284
26    2891
27    2502
28    2126
29    1857
30    1624
31    1398
32    1292
33    1181
34    1045
35     969
37     888
36     886
38     847
39     790
40     766
41     688
42     666
45     586
44     585
43     578
47     501
46     481
48     388
49     379
50     294
51     270
52     214
53     211
54     200
55     197
56     168
57     111
58     103
17      77
59      59
60      40
61      29
62      18
63      15
67       4
65       4
66       3
68       3
64       3
70       1
74       1
Name: Edad ingreso, dtype: int64

In [ ]:
# Calculate value counts for the specified column
value_counts = Demo['Edad ingreso'].value_counts()

# Identify values with counts greater than or equal to 100
values_to_keep = value_counts[value_counts >= 100].index

# Filter rows where the values in the specified column are in the values_to_keep list
Demo = Demo[Demo[column_name].isin(values_to_keep)]

In [ ]:
#Columna 'Edad salida'
column_name = 'Edad salida'
column_counts = Demo[column_name].value_counts()
column_counts

23    3610
24    3506
25    3446
22    3371
21    3361
26    3246
20    3063
27    2998
28    2756
29    2592
19    2230
30    2198
31    1998
32    1698
33    1630
34    1455
35    1373
36    1233
37    1179
38    1056
39    1041
40     922
42     835
41     822
43     819
44     742
45     738
46     684
47     682
48     598
18     578
49     561
60     499
50     494
51     453
52     403
53     321
54     284
55     276
56     249
57     225
58     211
61     177
59     162
62      85
64      45
63      42
65      40
66       5
68       3
67       1
Name: Edad salida, dtype: int64

In [ ]:
# Calculate value counts for the specified column
value_counts = Demo['Edad salida'].value_counts()

# Identify values with counts greater than or equal to 100
values_to_keep = value_counts[value_counts >= 200].index

# Filter rows where the values in the specified column are in the values_to_keep list
Demo = Demo[Demo[column_name].isin(values_to_keep)]

In [ ]:
#Columna Duración Meses
column_name = 'Duración Meses'
column_data = Demo[column_name]

# Calculate quartiles and IQR
Q1 = column_data.quantile(0.5)
Q3 = column_data.quantile(0.95)
IQR = Q3 - Q1

# Define the upper and lower bounds for outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify outliers
outliers = column_data[(column_data < lower_bound) | (column_data > upper_bound)]

# Create a new DataFrame without outliers
Demo = Demo[(column_data >= lower_bound) & (column_data <= upper_bound)]

In [ ]:
value_counts_sorted = Demo['Duración Meses'].value_counts().sort_values().head(n=50)
value_counts_sorted

301    1
308    1
304    2
285    2
295    2
309    2
290    2
288    2
283    2
289    2
3      3
277    3
273    3
306    3
297    3
269    4
260    4
282    4
299    4
313    4
296    4
293    4
307    4
254    5
291    5
298    5
302    5
267    5
271    5
281    5
270    5
294    5
280    5
312    5
303    5
292    6
287    6
268    6
310    6
278    6
245    6
274    6
279    7
266    7
246    7
243    7
311    7
284    7
207    8
252    8
Name: Duración Meses, dtype: int64

In [ ]:
# Calculate value counts for the specified column
value_counts = Demo['Duración Meses'].value_counts().sort_values()

# Identify values with counts greater than or equal to 100
values_to_keep = value_counts[value_counts >= 6].index

# Filter rows where the values in the specified column are in the values_to_keep list
Demo = Demo[Demo[column_name].isin(values_to_keep)]

In [ ]:
#Columna Duración Trimestres
column_name = 'Duración Trimestres'
column_data = Demo[column_name]

# Calculate quartiles and IQR
Q1 = column_data.quantile(0.5)
Q3 = column_data.quantile(0.95)
IQR = Q3 - Q1

# Define the upper and lower bounds for outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify outliers
outliers = column_data[(column_data < lower_bound) | (column_data > upper_bound)]

# Create a new DataFrame without outliers
Demo = Demo[(column_data >= lower_bound) & (column_data <= upper_bound)]

In [ ]:
value_counts_sorted = Demo['Duración Trimestres'].value_counts().sort_values().head(n=18)
value_counts_sorted

90     6
98     6
95     7
91     8
93    13
96    14
89    18
92    22
82    27
87    29
85    29
88    29
86    29
84    32
78    33
83    33
81    35
69    35
Name: Duración Trimestres, dtype: int64

In [ ]:
# Calculate value counts for the specified column
value_counts = Demo['Duración Trimestres'].value_counts().sort_values()

# Identify values with counts greater than or equal to 100
values_to_keep = value_counts[value_counts >= 35].index

# Filter rows where the values in the specified column are in the values_to_keep list
Demo = Demo[Demo[column_name].isin(values_to_keep)]

### Removal of Outliers in Categorical Columns




In [ ]:
#Columna Departamento
value_counts_sorted = Demo['Departamento'].value_counts().sort_values().head(n=113)
value_counts_sorted

OFICIOS VARIOS AIRE ACONDICIONADO     1
Surtido PLAN                          1
SOFT LINE                             1
Crédito y Cobranzas                   1
CAJAS CREDITO                         1
                                     ..
FINANZAS LIVERPOOL                    9
ACCESORIOS NIÑO                       9
COMPRAS DIVERSOS                     10
RECIBO CDI                           10
SOLUCIONES A MEDIDA                  11
Name: Departamento, Length: 113, dtype: int64

In [ ]:
# Step 1: Count the occurrences of each unique string value
value_counts = Demo['Departamento'].value_counts()

# Step 2: Filter string values with a count of less than 100
strings_to_drop = value_counts[value_counts < 11].index

# Step 3: Use boolean indexing to drop rows with these string values
Demo = Demo[~Demo['Departamento'].isin(strings_to_drop)]

In [ ]:
#Columna Desc Fun
value_counts_sorted = Demo['Desc Fun'].value_counts().sort_values().head(n=419)
value_counts_sorted

Analista Online Sfera        1
Coordinador Señalizacion     1
Director Centro Distribuc    1
Asesor Talento               1
Gerente Desarrollo Produc    1
                            ..
Tomador de Medidas           4
Encargado Limpieza           4
Gerente TI Infraestructur    4
Jefe de Estacionamiento A    4
Medico                       5
Name: Desc Fun, Length: 419, dtype: int64

In [ ]:
# Step 1: Count the occurrences of each unique string value
value_counts = Demo['Desc Fun'].value_counts()

# Step 2: Filter string values with a count of less than 100
strings_to_drop = value_counts[value_counts < 5].index

# Step 3: Use boolean indexing to drop rows with these string values
Demo = Demo[~Demo['Desc Fun'].isin(strings_to_drop)]

In [ ]:
#Columna Locación
value_counts_sorted = Demo['Locación'].value_counts().sort_values().head(n=89)
value_counts_sorted

Suburbia Los Cabos Patio        1
Sfera Villahermosa              1
ETAM Punto Valle                1
MAC Satélite                    1
Mac Mazatlán                    1
                               ..
Suburbia Patio Villahermosa     9
Adcon Torreon                   9
Banana Republic Coyoacán        9
Sfera Galerias Lag Torreon      9
Sfera Tezontle                 10
Name: Locación, Length: 89, dtype: int64

In [ ]:
# Step 1: Count the occurrences of each unique string value
value_counts = Demo['Locación'].value_counts()

# Step 2: Filter string values with a count of less than 100
strings_to_drop = value_counts[value_counts < 10].index

# Step 3: Use boolean indexing to drop rows with these string values
Demo = Demo[~Demo['Locación'].isin(strings_to_drop)]

In [ ]:
#Columna Área Personal
value_counts_sorted = Demo['Área Personal'].value_counts().sort_values()
value_counts_sorted

SBB Trainee Staff           1
SBB Trainee Operacio        3
Demo con Comision           3
Demo s/Comision Plnt        5
SBB Operación MT            7
SyD. Vendedor              20
Destajista                 26
SBB Subgerencia Oper       29
SBB Confidencial           37
Confidencial               38
M. Tiempo Pers.Gral.      171
SBB Staff                 309
SyD Pers.Gral.            439
MT. Vendedor              517
SBB Jefaturas Operac      735
Ejecutivo/ Coord.         981
Ejecutivo Quincenal      1084
SBB Operación           11050
TC. Vendedor            17439
Personal General        25512
Name: Área Personal, dtype: int64

In [ ]:
# Step 1: Count the occurrences of each unique string value
value_counts = Demo['Área Personal'].value_counts()

# Step 2: Filter string values with a count of less than 100
strings_to_drop = value_counts[value_counts < 39].index

# Step 3: Use boolean indexing to drop rows with these string values
Demo = Demo[~Demo['Área Personal'].isin(strings_to_drop)]

## Saving clean code

In [ ]:
# Guardar el DataFrame Historico en un nuevo archivo CSV
historico_limpiado_path= '/content/drive/Shareddrives/COLAB DATOS DE LIVERPOOL/Datos Liverpool/HistoricoLimpio.csv'
Historico.to_csv(historico_limpiado_path, index=False)

# Guardar el DataFrame Demo en un nuevo archivo CSV
demo_limpiado_path = '/content/drive/Shareddrives/COLAB DATOS DE LIVERPOOL/Datos Liverpool/DemoLimpio.csv'
Demo.to_csv(demo_limpiado_path, index=False)


## Data result

We created a brief summary using the pandas **.describe** function; this method provides descriptive statistics for all numerical columns in the dataset, including the count, mean, standard deviation, minimum and maximum values, and percentiles.

In [ ]:
# Obtener un resumen de los datos por columna
resumen = Demo.describe()
# Imprimir el resumen
print(resumen)

           Nº pers.      No Hijos          IAño          IMes  Edad ingreso  \
count  5.823700e+04  58237.000000  58237.000000  58237.000000  58237.000000   
mean   2.444655e+07      0.543194   2017.388980      7.068839     27.211275   
std    2.167634e+07      0.935874      3.465969      3.222012      8.623387   
min    1.397730e+05      0.000000   1999.000000      1.000000     18.000000   
25%    1.402284e+07      0.000000   2017.000000      4.000000     21.000000   
50%    1.487557e+07      0.000000   2018.000000      8.000000     24.000000   
75%    1.551921e+07      1.000000   2019.000000     10.000000     31.000000   
max    7.063535e+07      4.000000   2022.000000     12.000000     58.000000   

               SAño          SMes      Duracion    Antigüedad   Edad salida  \
count  58237.000000  58237.000000  58237.000000  58237.000000  58237.000000   
mean    2020.507358      6.091694      3.037983      2.534248     30.244621   
std        1.182473      3.496189      3.368264    

In [ ]:
# Obtener un resumen de los datos por columna
resumen = Historico.describe()
# Imprimir el resumen
print(resumen)

           Nº pers.           HAño           HMes           DAño  \
count  1.587630e+05  158763.000000  158763.000000  158763.000000   
mean   1.891917e+07    2017.776717       7.135561    2016.765115   
std    1.914448e+07       3.825906       3.416173       4.256934   
min    3.369000e+03    1998.000000       1.000000    1978.000000   
25%    1.186607e+07    2016.000000       5.000000    2016.000000   
50%    1.351383e+07    2019.000000       7.000000    2018.000000   
75%    1.483815e+07    2020.000000      10.000000    2019.000000   
max    7.063535e+07    2023.000000      12.000000    2022.000000   

                DMes       Duracion  Duración Días  
count  158763.000000  158763.000000  158763.000000  
mean        5.826024       1.178418     430.122396  
std         3.608649       1.420678     518.547432  
min         1.000000       0.334247     122.000000  
25%         2.000000       0.536986     196.000000  
50%         6.000000       0.830137     303.000000  
75%         9.00

In [ ]:
porcentaje_hijos_por_genero = Demo.groupby('Genero')['No Hijos'].mean()

# Encuentra el género con el mayor promedio de hijos
genero_con_mas_hijos = porcentaje_hijos_por_genero.idxmax()

# Calcula el porcentaje del género con más hijos
porcentaje_mas_hijos = (porcentaje_hijos_por_genero.max() / porcentaje_hijos_por_genero.sum()) * 100

# Muestra los resultados
print(f"Género con más hijos: {genero_con_mas_hijos}")
print(f"Porcentaje de género con más hijos: {porcentaje_mas_hijos:.2f}%")

Género con más hijos: Mujer
Porcentaje de género con más hijos: 58.79%


We determined the proportion of elements in each column of the *Demograficos* file.

We accomplished this using a loop to iterate through all the columns, rather than processing them one by one.

In [ ]:
columnasDemo= Demo.select_dtypes(include=['object']).columns

#Bucle en cada columna para determinar proporciones
for columna in columnasDemo:
    proporciones = Demo[columna].value_counts(normalize=True) * 100
    print(f"Proporción de elementos en la columna '{columna}':")
    print(proporciones)
    print("\n")

Proporción de elementos en la columna 'Fecha nacimiento':
1997-11-05    0.037777
1996-10-12    0.036060
1998-04-07    0.036060
1997-10-04    0.034342
1996-09-29    0.034342
                ...   
1974-04-21    0.001717
1976-11-04    0.001717
1965-05-09    0.001717
1980-02-26    0.001717
1995-06-07    0.001717
Name: Fecha nacimiento, Length: 13223, dtype: float64


Proporción de elementos en la columna 'Genero':
Mujer     51.982417
Hombre    48.017583
Name: Genero, dtype: float64


Proporción de elementos en la columna 'Desc Medida':
Baja                            79.217679
Baja Suburbia                   20.777169
Baja por Renuncia Voluntaria     0.005151
Name: Desc Medida, dtype: float64


Proporción de elementos en la columna 'CP Vivienda':
Sin Dato    1.887116
54680       1.231176
2770        1.186531
5348        0.733211
54955       0.630184
              ...   
59492       0.001717
1856        0.001717
2125        0.001717
54844       0.001717
97150       0.001717
Name: CP Vivien

We determined the proportion of elements in each column of the *Historicos* file.

We accomplished this using a loop to iterate through all the columns, rather than processing them one by one.

In [ ]:
columnasHistorico= Historico.select_dtypes(include=['object']).columns

for columna in columnasHistorico:
    proporciones = Historico[columna].value_counts(normalize=True) * 100
    print(f"Proporción de elementos en la columna '{columna}':")
    print(proporciones)
    print("\n")

Proporción de elementos en la columna 'Desc Soc':
Almacenes Com Liver SA de    23.430522
Operadora Comercial Liver    17.717604
Operadora Suburbia SRL CV     9.308844
Distribuidora Liverpool,      7.434982
Bodegas Liverpool, S.A. d     6.048009
Operadora Merc Liv SA         5.636074
Operadora de Alm Liv SA       5.357671
Tiendas Depart. Liver. SA     5.008094
Servicios Liverpool S.A.      3.985815
Almacenadora Liverpool SA     2.777725
Suburbia S. de R.L. de CV     2.165492
Liverpool Provincia, S.A.     1.642070
Liverpool México, S.A. de     1.446811
Operadora Serv Suburbia       1.226986
Grandes Alm Liverpool, SA     1.206830
Adconins, S.A. de C.V.        1.089045
Operadora Liv México SA       0.879298
Operadora Coml las Nvas F     0.667662
Frasyr, S.A. de C.V.          0.667032
Operadora Sfera México SA     0.575071
Comifral, S.A. de C.V.        0.418233
Adcon, S.A. de C.V.           0.207227
Liverpool Monterrey, S.A.     0.188961
Adconquer, S.A. de C.V.       0.168805
Adcoapa, S.A. 

The highest number of children belonging to an employee who has resigned or been dismissed.

We achieved this by finding the maximum value in the "No. de hijos" column.

In [ ]:
MaxHijos= Demo['No Hijos'].max()
print(f"La mayor cantidad de hijos de un empleado que ha renunciado es: {MaxHijos}")

La mayor cantidad de hijos de un empleado que ha renunciado es: 4


The earliest recorded date of an employee resignation.
We obtained this using the `min` function, applying it to the columns from the historical and demographic files.

In [ ]:
fechaAntiguaDemo= Demo['Fecha Salida'].min()

Historico['Fecha Salida'] = pd.to_datetime(Historico['Fecha Salida'])
fechaAntiguaHistorico= Historico['Fecha Salida'].min()

# Imprime la fecha más antigua
print(f"La fecha más antigua de renuncia de un empleado en el documento Demografico es: {fechaAntiguaDemo}")
print(f"La fecha más antigua de renuncia de un empleado en el documento Historico es: {fechaAntiguaHistorico}")


La fecha más antigua de renuncia de un empleado en el documento Demografico es: 2019-01-01 00:00:00
La fecha más antigua de renuncia de un empleado en el documento Historico es: 1998-01-30 00:00:00


Latest recorded date of an employee's resignation.
We used the max() function.

In [ ]:
# Encuentra la última fecha de renuncia
fechaRecienteDemo= Demo['Fecha Salida'].max()
fechaRecienteHistorico= Historico['Fecha Salida'].max()

# Imprime la última fecha
print(f"La última fecha de renuncia de un empleado en el documento Demografico es: {fechaRecienteDemo}")
print(f"La última fecha de renuncia de un empleado en el documento Historico es: {fechaRecienteHistorico}")

La última fecha de renuncia de un empleado en el documento Demografico es: 2022-12-31 00:00:00
La última fecha de renuncia de un empleado en el documento Historico es: 2023-03-13 00:00:00


The job role (salesperson, manager, assistant, etc.) with the highest number of departures; this analysis was performed on both the demographics table and the historical data.

Here we can see that this corresponds to role 897, which is "Vendedor Cajero" in both files.

In [ ]:
display(Demo)

,Nº pers.,Fecha nacimiento,Genero,No Hijos,Fecha ingreso,IAño,IMes,Edad ingreso,Fecha Salida,SAño,...,CP Trabajo,Locación,Desc Fun,GroPer,Área Personal,Departamento,Unidad,Duración Días,Duración Meses,Duración Trimestres
28,15212625,2000-07-20,Hombre,0,2019-11-27,2019,11,19,2022-08-18,2022,...,1140,CeDis Regional Tacubaya,Personal Operativo,Planta No Sind.,Personal General,ENVIOS,Supervision Recepcion CD,995,33,11
29,15472429,1978-12-21,Hombre,0,2021-04-28,2021,4,42,2022-03-01,2022,...,1140,CeDis Regional Tacubaya,Recepcion Mercancia,Eventual No Sind.,Personal General,PAQUETERIA,Jefatura Paqueteria I CD,307,11,4
37,15375754,1982-02-10,Hombre,0,2020-10-19,2020,10,38,2021-06-08,2021,...,1140,CeDis Regional Tacubaya,Auxiliar Operador,Eventual No Sind.,Personal General,PLANTILLA EVENTUAL,Pool Eventuales B Tacubay,232,8,3
39,11915238,1992-04-24,Hombre,0,2011-03-01,2011,3,18,2022-02-16,2022,...,1140,CeDis Regional Tacubaya,Recepcion Mercancia,Eventual No Sind.,Personal General,PLANTILLA EVENTUAL,Pool Eventuales B Tacubay,4005,131,44
40,13792355,1995-12-04,Hombre,1,2016-09-09,2016,9,20,2022-06-16,2022,...,1140,CeDis Regional Tacubaya,Personal Operativo,Vía Planta No Sind.,Personal General,RECEPCION,Supervision Recepcion CD,2106,69,23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117141,14057601,1997-04-05,Mujer,0,2017-04-17,2017,4,20,2019-03-23,2019,...,5109,Sfera Veracruz Veramar,Vendedor Boutique Sfera,Planta No Sind.,TC. Vendedor,SFERA,Ventas Sfera BT Veracruz,705,23,8
117144,14503279,1996-07-18,Hombre,0,2018-04-04,2018,4,21,2019-04-04,2019,...,5109,Sfera Tlaquepaque,Vendedor Boutique Sfera,Planta No Sind.,MT. Vendedor,BOUTIQUES,Ventas Sfera BT Tlaquepaq,365,12,4
117145,14583940,1996-01-01,Mujer,0,2018-06-06,2018,6,22,2019-01-29,2019,...,5348,Sfera Playa del Carmen,Vendedor Boutique Sfera,Planta No Sind.,MT. Vendedor,BOUTIQUES,Ventas Sfera BT Playa del,237,7,3
117149,14864493,1990-08-30,Hombre,0,2019-01-18,2019,1,28,2020-01-14,2020,...,5348,Sfera Guadalajara Gran Plaza,Vendedor Boutique Sfera,Planta No Sind.,TC. Vendedor,BOUTIQUES,Ventas Sfera Gran Plaza,361,12,4


In [ ]:
FrecuenciaFuncionHist= Historico['Desc Función'].value_counts()
FrecuenciaFuncionDemo= Demo['Desc Fun'].value_counts()

# Encontrar la función con más bajas
funcionConBajasHist=FrecuenciaFuncionHist.idxmax()
funcionConBajasDemo=FrecuenciaFuncionDemo.idxmax()

# Imprimir la función con más bajas
print(f"La función con más bajas en el documento Historico es: {funcionConBajasHist}")
print(f"La función con más bajas en el documento Demografico es: {funcionConBajasDemo}")

La función con más bajas en el documento Historico es: Vendedor Cajero
La función con más bajas en el documento Demografico es: Vendedor Cajero


Job type (salesperson, manager, assistant, etc.) with the FEWEST absences.

> In the Historicos Data section, it corresponds to job title #4928, which is "consultor Validación".

> In the Demograficos section, it corresponds to "Jefe Administración e Int".

In [ ]:
FrecuenciaFuncionHist= Historico['Desc Función'].value_counts()
FrecuenciaFuncionDemo= Demo['Desc Fun'].value_counts()

# Encontrar la función con más bajas
funcionConMenosHist=FrecuenciaFuncionHist.idxmin()
funcionConMenosDemo=FrecuenciaFuncionDemo.idxmin()

# Imprimir la función con más bajas
print(f"La función con menos bajas en el documento Historico es: {funcionConMenosHist}")
print(f"La función con menos bajas en el documento Demografico es: {funcionConMenosDemo}")

La función con menos bajas en el documento Historico es: Coordinador Pick&Mix
La función con menos bajas en el documento Demografico es: Promotor no Ventas


Number of departures from Liverpool and Suburbia.
There was no specific column indicating which records belonged to Liverpool or Suburbia, so we searched for these terms as keywords within the "Locación" column.

> In this code, we define the keywords "liverpool" and "Suburbia." Then, we create boolean masks using `str.contains()` to search for these keywords in the "Locación" column. We use `case=False` to make the search case-insensitive and `na=False` to ensure NaN values ​​are not included.

> Next, we count the number of employees found for each keyword and store the results in the variables `conteo_liverpool` and `conteo_suburbia`. Finally, we print the results to show how many employees are from Liverpool and how many are from Suburbia.

In [ ]:
palabraLiverpool = "Liverpool"
palabraSuburbia= "Suburbia"

# Crear máscaras booleanas para las palabras clave
Liverpool= Demo['Locación'].str.contains(palabraLiverpool, case=False, na=False)
Suburbia = Demo['Locación'].str.contains(palabraSuburbia, case=False, na=False)

LiverpoolH= Historico['Locación'].str.contains(palabraLiverpool, case=False, na=False)
SuburbiaH= Historico['Locación'].str.contains(palabraSuburbia, case=False, na=False)

# Contar cuántos empleados son de Liverpool y cuántos son de Suburbia
empleados_liverpool = Demo[Liverpool]
empleados_suburbia = Demo[Suburbia]
empleados_liverpoolH = Historico[LiverpoolH]
empleados_suburbiaH= Historico[SuburbiaH]

# Obtener el conteo de empleados
conteo_liverpool = len(empleados_liverpool)
conteo_suburbia = len(empleados_suburbia)
conteo_liverpoolH = len(empleados_liverpoolH)
conteo_suburbiaH = len(empleados_suburbiaH)

# Imprimir los resultados
print(f"Empleados de Liverpool en Demograficos: {conteo_liverpool}")
print(f"Empleados de Suburbia en Demograficos: {conteo_suburbia}")
print(f"Empleados de Liverpool en Historicos: {conteo_liverpool}")
print(f"Empleados de Suburbia en Historicos: {conteo_suburbia}")

Empleados de Liverpool en Demograficos: 33607
Empleados de Suburbia en Demograficos: 12027
Empleados de Liverpool en Historicos: 33607
Empleados de Suburbia en Historicos: 12027


Number of "Vendedores hombre" who have resigned.
There is no specific column with those characteristics; it must be calculated using the "Género" and "Desc Fun" columns.

> In this code, we first filter the `Demo` DataFrame to select rows where the "Género" column is "Hombre" and the "Desc Fun" column contains the word "Vendedor" (the keyword search in "Desc Fun" is performed using `str.contains`). Then, we count the number of rows that meet this condition to obtain the desired result.

In [ ]:
#fila en donde busco por palabra clave y me revisa las filas de hombres que cumplen con ser vendedores.
HombreVendedor= Demo[(Demo['Genero']=='Hombre') & (Demo['Desc Fun'].str.contains('Vendedor', case=False))]

#aqui ya solo lo suma
NumHombreVendedor= len(HombreVendedor)
print(f"Número de Vendedores hombres que han renunciado: {NumHombreVendedor}")

Número de Vendedores hombres que han renunciado: 6964


Number of distinct dates across the various files provided.
For this step, we use the concatenate function to combine both dates and then count them; here, we are taking into account that a single person may have both a start date and an end date.

In [ ]:
#ARCHIVO DEMOGRAFICO
# Concatenar las columnas "Fecha de entrada" y "Fecha de Salida"
fechasDemo= pd.concat([Demo['Fecha ingreso'], Demo['Fecha Salida']])
num_fechas_demo = len(fechasDemo.unique())
print(f"Número de fechas distintas en Demo: {num_fechas_demo}")

Número de fechas distintas en Demo: 5410


In [ ]:
#ARCHIVO HISTORICO
fechasHistorico = pd.concat([Historico['Desde'], Historico['Fecha Salida']])
num_fechas_historico = len(fechasHistorico.unique())
print(f"Número de fechas distintas en Historico: {num_fechas_historico}")

Número de fechas distintas en Historico: 6610


### Renunciations count by category

In [ ]:
cuenta_por_cat(Historico,'DAño')
cuenta_por_cat(Historico,'HAño')
cuenta_por_cat(Historico,'Departamento')
cuenta_por_cat(Historico, 'Desc Función')

      Conteo  Porcentaje
DAño                    
1978       2    0.001260
1979       5    0.003149
1980       7    0.004409
1981      16    0.010078
1982      11    0.006929
1983      15    0.009448
1984      17    0.010708
1985      15    0.009448
1986      17    0.010708
1987      18    0.011338
1988      16    0.010078
1989      28    0.017636
1990      23    0.014487
1991      32    0.020156
1992      24    0.015117
1993      32    0.020156
1994      45    0.028344
1995      32    0.020156
1996      51    0.032123
1997      84    0.052909
1998     129    0.081253
1999     105    0.066136
2000    1010    0.636168
2001     392    0.246909
2002     542    0.341389
2003     527    0.331941
2004    1124    0.707974
2005     810    0.510194
2006    1030    0.648766
2007    1167    0.735058
2008    1424    0.896934
2009    1215    0.765292
2010    3578    2.253674
2011    2162    1.361778
2012    2710    1.706947
2013    6646    4.186114
2014    4308    2.713479
2015    8630    5.435775
